In [4]:
import sys
import os

print("Python version:")
print(sys.version)

print("\nCurrent working directory:")
print(os.getcwd())

Python version:
3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]

Current working directory:
C:\Users\86183\Desktop\chooser-option-pricing-main\notebooks


In [5]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("Project root:")
print(project_root)

print("\nModel file:")
print(project_root / "model" / "bsm_chooser.py")

Project root:
C:\Users\86183\Desktop\chooser-option-pricing-main

Model file:
C:\Users\86183\Desktop\chooser-option-pricing-main\model\bsm_chooser.py


In [6]:
from model.bsm_chooser import simulate_chooser_option

print("BSM Chooser model imported successfully.")

BSM Chooser model imported successfully.


In [7]:
# ==========================================
# Week 3 - Paper Baseline Parameters
# ==========================================

S0 = 156.7
risk_free_rate = 0.0015
sigma = 0.282
strike = 150.0
T1 = 0.5
T2 = 1.0

n_simulations = 10000
seed = 42

results = simulate_chooser_option(
    S0=S0,
    risk_free_rate=risk_free_rate,
    sigma=sigma,
    strike=strike,
    T1=T1,
    T2=T2,
    n_simulations=n_simulations,
    seed=seed,
)

print("Paper Baseline Parameters")
print("-" * 40)
print(f"S0       = ${S0:.2f}")
print(f"r        = {risk_free_rate:.2%}")
print(f"sigma    = {sigma:.2%}")
print(f"K        = ${strike:.2f}")
print(f"T1       = {T1:.2f} years")
print(f"T2       = {T2:.2f} years")
print(f"Simulations = {n_simulations}")
print(f"Seed         = {seed}")

print("\nSimulation Result")
print("-" * 40)
print(f"Chooser Option Price = ${results['Chooser_Price']:.4f}")

Paper Baseline Parameters
----------------------------------------
S0       = $156.70
r        = 0.15%
sigma    = 28.20%
K        = $150.00
T1       = 0.50 years
T2       = 1.00 years
Simulations = 10000
Seed         = 42

Simulation Result
----------------------------------------
Chooser Option Price = $29.6973


In [8]:
import numpy as np

call_count = np.sum(results["Choice"] == "CALL")
put_count = np.sum(results["Choice"] == "PUT")

print("Choice Statistics")
print("-" * 40)
print(f"CALL choices = {call_count}")
print(f"PUT choices  = {put_count}")
print(f"CALL ratio   = {call_count / n_simulations:.2%}")
print(f"PUT ratio    = {put_count / n_simulations:.2%}")

print("\nPayoff Statistics")
print("-" * 40)
print(f"Mean payoff = ${np.mean(results['Payoff']):.4f}")
print(f"Max payoff  = ${np.max(results['Payoff']):.4f}")
print(f"Zero payoff = {np.mean(results['Payoff'] == 0):.2%}")

Choice Statistics
----------------------------------------
CALL choices = 5470
PUT choices  = 4530
CALL ratio   = 54.70%
PUT ratio    = 45.30%

Payoff Statistics
----------------------------------------
Mean payoff = $29.7419
Max payoff  = $321.0886
Zero payoff = 25.93%


In [9]:
import pandas as pd

validation_table = pd.DataFrame({
    "Simulation": range(1, 11),
    "S_T1": results["S1"][:10],
    "Decision": results["Choice"][:10],
    "S_T2": results["S2"][:10],
    "Payoff": results["Payoff"][:10],
    "Discounted_Payoff": results["Discounted_Payoff"][:10],
})

validation_table


,Simulation,S_T1,Decision,S_T2,Payoff,Discounted_Payoff
0,1,163.361247,CALL,165.999264,15.999264,15.975283
1,2,124.938928,PUT,146.640346,3.359654,3.354618
2,3,178.545809,CALL,130.599761,0.000000,0.000000
3,4,185.444310,CALL,171.869247,21.869247,21.836468
4,5,104.183911,PUT,120.586011,29.413989,29.369901
5,6,118.574588,PUT,117.859371,32.140629,32.092454
6,7,157.699930,CALL,207.297601,57.297601,57.211719
7,8,144.335735,PUT,103.416599,46.583401,46.513579
8,9,153.216496,CALL,107.310621,0.000000,0.000000
9,10,129.684125,PUT,148.331456,1.668544,1.666043


In [10]:
# ==========================================
# Paper Validation Summary
# ==========================================

validation_summary = pd.DataFrame({
    "Parameter": [
        "Initial Stock Price (S0)",
        "Risk-free Rate (r)",
        "Volatility (sigma)",
        "Strike Price (K)",
        "Decision Date (T1)",
        "Maturity (T2)",
        "Monte Carlo Simulations",
        "Random Seed",
        "Chooser Option Price"
    ],
    "Value": [
        S0,
        risk_free_rate,
        sigma,
        strike,
        T1,
        T2,
        n_simulations,
        seed,
        results["Chooser_Price"]
    ]
})

validation_summary

,Parameter,Value
0,Initial Stock Price (S0),156.700000
1,Risk-free Rate (r),0.001500
2,Volatility (sigma),0.282000
3,Strike Price (K),150.000000
4,Decision Date (T1),0.500000
5,Maturity (T2),1.000000
6,Monte Carlo Simulations,10000.000000
7,Random Seed,42.000000
8,Chooser Option Price,29.697297


In [11]:
print("Model Validation")
print("=" * 50)

print("✓ BSM two-stage simulation completed")
print("✓ Decision rule implemented: S_T1 > K → CALL, otherwise PUT")
print("✓ Final payoff calculated using selected option")
print("✓ Payoff discounted to present value")
print(f"✓ Reproduced Chooser Option Price: ${results['Chooser_Price']:.4f}")

Model Validation
✓ BSM two-stage simulation completed
✓ Decision rule implemented: S_T1 > K → CALL, otherwise PUT
✓ Final payoff calculated using selected option
✓ Payoff discounted to present value
✓ Reproduced Chooser Option Price: $29.6973


In [12]:
import pandas as pd

data_path = project_root / "data" / "processed" / "jpm_features.csv"

df = pd.read_csv(data_path)
df["Date"] = pd.to_datetime(df["Date"])

print("Dataset shape:", df.shape)
print("Start date:", df["Date"].min().date())
print("End date:", df["Date"].max().date())

print("\nLatest observations:")
print(
    df[
        [
            "Date",
            "Adj Close",
            "DGS10",
            "Rolling_Volatility_5D",
            "Rolling_Volatility_20D",
            "Rolling_Volatility_60D",
        ]
    ].tail()
)

Dataset shape: (1701, 23)
Start date: 2018-03-29
End date: 2024-12-31

Latest observations:
           Date  Adj Close  DGS10  Rolling_Volatility_5D  \
1696 2024-12-24     233.96   4.59               0.021595   
1697 2024-12-26     234.77   4.58               0.007496   
1698 2024-12-27     232.86   4.62               0.011301   
1699 2024-12-30     231.08   4.55               0.010078   
1700 2024-12-31     231.45   4.58               0.010029   

      Rolling_Volatility_20D  Rolling_Volatility_60D  
1696                0.011785                0.019999  
1697                0.011839                0.019813  
1698                0.011935                0.019864  
1699                0.012002                0.019841  
1700                0.011694                0.019372  


In [13]:
# ==========================================
# Updated JPM Real-World Parameters
# ==========================================

latest = df.iloc[-1]

S0_updated = latest["Adj Close"]
risk_free_rate_updated = latest["DGS10"] / 100

daily_volatility_60d = latest["Rolling_Volatility_60D"]
sigma_updated = daily_volatility_60d * (252 ** 0.5)

strike_updated = 150.0
T1_updated = 0.5
T2_updated = 1.0

print("Updated JPM BSM Parameters")
print("=" * 50)

print(f"Valuation Date       = {latest['Date'].date()}")
print(f"S0                   = ${S0_updated:.2f}")
print(f"Risk-free Rate       = {risk_free_rate_updated:.2%}")
print(f"60D Daily Volatility = {daily_volatility_60d:.4f}")
print(f"Annualized Volatility= {sigma_updated:.2%}")
print(f"Strike Price (K)     = ${strike_updated:.2f}")
print(f"T1                   = {T1_updated:.2f} years")
print(f"T2                   = {T2_updated:.2f} years")

Updated JPM BSM Parameters
Valuation Date       = 2024-12-31
S0                   = $231.45
Risk-free Rate       = 4.58%
60D Daily Volatility = 0.0194
Annualized Volatility= 30.75%
Strike Price (K)     = $150.00
T1                   = 0.50 years
T2                   = 1.00 years


In [14]:
# ==========================================
# Updated JPM Chooser Option Pricing
# ==========================================

updated_results = simulate_chooser_option(
    S0=S0_updated,
    risk_free_rate=risk_free_rate_updated,
    sigma=sigma_updated,
    strike=strike_updated,
    T1=T1_updated,
    T2=T2_updated,
    n_simulations=10000,
    seed=42,
)

updated_call_count = np.sum(
    updated_results["Choice"] == "CALL"
)

updated_put_count = np.sum(
    updated_results["Choice"] == "PUT"
)

print("Updated JPM Chooser Option")
print("=" * 50)

print(f"Valuation Date       = {latest['Date'].date()}")
print(f"S0                   = ${S0_updated:.2f}")
print(f"Risk-free Rate       = {risk_free_rate_updated:.2%}")
print(f"Annualized Volatility= {sigma_updated:.2%}")
print(f"Strike Price         = ${strike_updated:.2f}")
print(f"T1                   = {T1_updated:.2f} years")
print(f"T2                   = {T2_updated:.2f} years")

print("\nSimulation Results")
print("-" * 50)
print(f"Simulations           = 10,000")
print(f"CALL choices           = {updated_call_count}")
print(f"PUT choices            = {updated_put_count}")
print(f"CALL ratio             = {updated_call_count / 10000:.2%}")
print(f"PUT ratio              = {updated_put_count / 10000:.2%}")
print(
    f"Updated Chooser Price  = "
    f"${updated_results['Chooser_Price']:.4f}"
)

Updated JPM Chooser Option
Valuation Date       = 2024-12-31
S0                   = $231.45
Risk-free Rate       = 4.58%
Annualized Volatility= 30.75%
Strike Price         = $150.00
T1                   = 0.50 years
T2                   = 1.00 years

Simulation Results
--------------------------------------------------
Simulations           = 10,000
CALL choices           = 9762
PUT choices            = 238
CALL ratio             = 97.62%
PUT ratio              = 2.38%
Updated Chooser Price  = $90.1773


In [15]:
# ==========================================
# Week 3 Final Validation Summary
# ==========================================

validation_results = pd.DataFrame({
    "Model": [
        "Paper Baseline",
        "Updated JPM"
    ],
    "Valuation Date": [
        "Paper Parameters",
        str(latest["Date"].date())
    ],
    "S0": [
        S0,
        S0_updated
    ],
    "Risk-free Rate": [
        risk_free_rate,
        risk_free_rate_updated
    ],
    "Volatility": [
        sigma,
        sigma_updated
    ],
    "Strike": [
        strike,
        strike_updated
    ],
    "T1": [
        T1,
        T1_updated
    ],
    "T2": [
        T2,
        T2_updated
    ],
    "Simulations": [
        n_simulations,
        10000
    ],
    "Call Ratio": [
        np.mean(results["Choice"] == "CALL"),
        np.mean(updated_results["Choice"] == "CALL")
    ],
    "Put Ratio": [
        np.mean(results["Choice"] == "PUT"),
        np.mean(updated_results["Choice"] == "PUT")
    ],
    "Chooser Price": [
        results["Chooser_Price"],
        updated_results["Chooser_Price"]
    ]
})

validation_results

,Model,Valuation Date,S0,Risk-free Rate,Volatility,Strike,T1,T2,Simulations,Call Ratio,Put Ratio,Chooser Price
0,Paper Baseline,Paper Parameters,156.70,0.0015,0.282000,150.0,0.5,1.0,10000,0.5470,0.4530,29.697297
1,Updated JPM,2024-12-31,231.45,0.0458,0.307513,150.0,0.5,1.0,10000,0.9762,0.0238,90.177317


In [16]:
# ==========================================
# Formatted Validation Table
# ==========================================

formatted_results = validation_results.copy()

formatted_results["S0"] = formatted_results["S0"].map(
    lambda x: f"${x:.2f}"
)

formatted_results["Risk-free Rate"] = formatted_results["Risk-free Rate"].map(
    lambda x: f"{x:.2%}"
)

formatted_results["Volatility"] = formatted_results["Volatility"].map(
    lambda x: f"{x:.2%}"
)

formatted_results["Strike"] = formatted_results["Strike"].map(
    lambda x: f"${x:.2f}"
)

formatted_results["Call Ratio"] = formatted_results["Call Ratio"].map(
    lambda x: f"{x:.2%}"
)

formatted_results["Put Ratio"] = formatted_results["Put Ratio"].map(
    lambda x: f"{x:.2%}"
)

formatted_results["Chooser Price"] = formatted_results["Chooser Price"].map(
    lambda x: f"${x:.4f}"
)

In [17]:
formatted_results

,Model,Valuation Date,S0,Risk-free Rate,Volatility,Strike,T1,T2,Simulations,Call Ratio,Put Ratio,Chooser Price
0,Paper Baseline,Paper Parameters,$156.70,0.15%,28.20%,$150.00,0.5,1.0,10000,54.70%,45.30%,$29.6973
1,Updated JPM,2024-12-31,$231.45,4.58%,30.75%,$150.00,0.5,1.0,10000,97.62%,2.38%,$90.1773


# Week 3 Conclusion

The BSM-based two-stage chooser option pricing model was successfully implemented and validated.

Using the paper baseline parameters, the Monte Carlo simulation produced a chooser option price of **USD 29.6973** with 10,000 simulations. The simulated decision outcomes were 54.70% Call choices and 45.30% Put choices.

Using updated JPM market data as of **2024-12-31**, the model used an adjusted closing price of **USD 231.45**, a 10-Year Treasury yield of **4.58%**, and an annualized 60-day volatility of **30.75%**. The resulting chooser option price was **USD 90.1773**. Under these parameters, 97.62% of simulated paths selected the Call and 2.38% selected the Put.

The difference between the two simulated prices reflects the different market inputs used in the two parameter sets, particularly the higher JPM stock price, risk-free rate, and volatility in the updated parameter set. These results demonstrate that the implemented model responds to changes in market inputs and can be applied to updated JPM data.

The model, validation notebook, and parameter configuration file together complete the main Week 3 deliverables.